# Investigation of RAB29 p.R175K haplotype and LD with RAB29 GWAS hit

- Project: RAB GTPase gene investigation and burden analysis across populations with Parkinson’s disease
- Version: Python/3.10.19
- Last Updated: 10-AUG-2026

# Notebook Overview
- 1 RAB29 - p.R175K Haplotype and LD analysis with other variants
  - 1.1 Loading Python libraries
  - 1.2 Set Paths
  - 1.3 Defining functions
  - 1.4 Haplotype analysis in RAB29 p.R175K variant carriers
  - 1.5 LD of RAB29 p.R175K with RAB29 GWAS hit
  - 1.6 Regression analysis of RAB29 p.R175K with RAB29 GWAS hit

# *RAB29* - p.R175K Haplotype and LD analysis with other variants

## Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for Dataframe functionality
import pandas as pd

# Numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

# Use pathlib for file path manipulation
import pathlib

# Import Sys
import sys as sys

# Import Seaborn
import seaborn as sns

# Install plotnine: a ggplot2-compatible Python plotting package
from plotnine import *

# Import Glob
import glob

# Import Re
import re

# Import CYVCF2
#from cyvcf2 import VCF

# Import tools for meta-analysis
from scipy import stats
from scipy.stats import fisher_exact

# Import collections
from collections import defaultdict, deque

# Import Tempfile
import tempfile

# Import Shutil
import shutil

# Always show all columns in a Pandas DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('future.no_silent_downcasting', True)

## Set paths

In [ ]:
## GP2 v11.0
REL11_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release11')

WGS_DATA_PATH = pathlib.Path(REL11_PATH, 'wgs/deepvariant_joint_calling/plink')
WGS_RELATED_PATH = pathlib.Path(REL11_PATH, 'wgs/deepvariant_joint_calling/related_samples')
CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/master_key_release11_final_vwb.csv')

## Defining functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))

## Haplotype analysis in *RAB29* p.R175K variant carriers

In [ ]:
# Create separate lists of all RAB29 R175K carriers (FID and IID)

# Create output directory
!mkdir /home/jupyter/results/R11/Haplotype/

# Input files of EUR RAB29 R175K variant carriers
RAB29_R175K_csv_file = '/home/jupyter/results/R11/master_key_release11_final_vwb_RAB29_R175K.csv'
RAB29_R175K_fam_path = '/home/jupyter/results/R11/RAB29_WGS_EUR/RAB29.fam'

# Read the .fam file
RAB29_R175K_fam_df = pd.read_csv(RAB29_R175K_fam_path, sep='\s+', header=None,
                                 names=['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENO'])

# Read the carrier CSV
carrier_df = pd.read_csv(RAB29_R175K_csv_file)
carrier_ids = set(carrier_df['GP2ID'])

# Match IDs from .fam
matched_fam = RAB29_R175K_fam_df[RAB29_R175K_fam_df['IID'].isin(carrier_ids)][['FID', 'IID']]

# Create output path and save output
out_path = f'/home/jupyter/results/R11/Haplotype/RAB29_R175K_IDs.txt'
matched_fam.to_csv(out_path, sep='\t', header=False, index=False)
print(f"{out_path}: {len(matched_fam)} IDs included.")


# Create separate lists of all RAB29 R175K carriers (FID.IID)

# Concatenate FID and IID with a dot
concatenated = matched_fam['FID'] + '.' + matched_fam['IID']

# Save to output file
out_path2 = f'/home/jupyter/results/R11/Haplotype/RAB29_R175K_IDs_FID.IID.txt'
concatenated.to_csv(out_path2, sep='\t', header=False, index=False)

# Print summary
print(f" {out_path2}: {len(concatenated)} IDs included.")

In [ ]:
# Prepare file ±1Mb around the RAB29 R175K variant
# EUR
WORK_DIR = f'/home/jupyter/results/R11/RAB29_WGS_EUR'
OUT_DIR = f'/home/jupyter/results/R11/Haplotype'

! /home/jupyter/tools/plink/plink2 \
--pfile {WGS_DATA_PATH}/EUR/chr1_EUR_release11  \
--geno 0.05 \
--make-pgen \
--max-alleles 2 \
--out {WORK_DIR}/chr1.qc_EUR_release11

! /home/jupyter/tools/plink/plink2 \
--pfile {WORK_DIR}/chr1.qc_EUR_release11 \
--chr 1 \
--from-bp 204770430 \
--to-bp 206770430 \
--make-bed \
--export vcf id-delim=. \
--out {OUT_DIR}/RAB29_EUR_1MB

!rm {WORK_DIR}/chr1.qc_EUR_release11*

In [ ]:
%%bash
# Prepare VCF file to match fasta file 

# Set working directory
cd /home/jupyter/results/R11/Haplotype

# Define input VCF file (adjust the filename as needed)
vcf="RAB29_EUR_1MB.vcf"
prep_vcf="RAB29_EUR_1MB.prep.vcf"

# Modify contig lines and chromosome names to match FASTA reference
awk '
BEGIN { OFS="\t" }
/^##contig=<ID=[0-9XYM]/ { sub(/^##contig=<ID=/, "##contig=<ID=chr"); print; next }
/^#/ { print; next }
{ $1 = "chr"$1; print }
' "$vcf" > "$prep_vcf"

# Reference FASTA file
FASTA="/home/jupyter/tools/reference/chr1.fa"

# Normalize and left-align alleles using bcftools
norm_vcf="RAB29_EUR_1MB.norm.vcf.gz"
bcftools norm -f "$FASTA" -m -any "$prep_vcf" -Oz -o "$norm_vcf"
tabix -f "$norm_vcf"

In [ ]:
%%bash

# Correct the VCF file to match the Beagle map file
cd /home/jupyter/results/R11/Haplotype
norm_vcf="RAB29_EUR_1MB.norm.vcf.gz"
output="RAB29_EUR_1MB.norm.nochr.vcf.gz"
    
bcftools annotate \
    --rename-chrs <(awk '{print $1, substr($1,4)}' /home/jupyter/tools/reference/chr1.fa.fai) \
    "$norm_vcf" -Oz -o "$output"
    
tabix -f "$output"

In [ ]:
%%bash

# Phasing with Beagle
cd /home/jupyter/results/R11/Haplotype
java -Xmx4g -jar /home/jupyter/tools/beagle/beagle.28Jun21.220.jar \
  gt=RAB29_EUR_1MB.norm.nochr.vcf.gz \
  out=RAB29_EUR_1MB.phased \
  nthreads=4 \
  map=/home/jupyter/tools/beagle/plink.chr1.GRCh38.map

tabix -p vcf RAB29_EUR_1MB.phased.vcf.gz
gunzip -k RAB29_EUR_1MB.phased.vcf.gz

In [ ]:
%%bash

# Prepare list of variants with MAF>0.3
cd /home/jupyter/results/R11/Haplotype
/home/jupyter/tools/plink/plink1.9 \
--bfile RAB29_EUR_1MB \
--maf 0.3 \
--make-bed \
--out RAB29_EUR_1MB_filtered_MAF

awk '{print $2}' RAB29_EUR_1MB_filtered_MAF.bim > RAB29_EUR_1MB_filtered_MAF_SNP_IDs.txt

In [ ]:
# Add R175K variant to the list of SNPs with MAF>0.3
# Original file path
original_file = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_filtered_MAF_SNP_IDs.txt"

# Add R175K SNP
new_snp = "chr1:205770430:C:T"

# New file path
new_file = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_filtered_MAF_R175K_SNP_IDs.txt"

# Read original SNPs
with open(original_file, "r") as file:
    snps = file.read().splitlines()

# Append the new SNP
snps.append(new_snp)

# Write to new file
with open(new_file, "w") as file:
    for snp in snps:
        file.write(f"{snp}\n")

print(f"New file created: {new_file} with SNP {new_snp} added.")

In [ ]:
from cyvcf2 import VCF, Writer

# Input/output files
input_vcf = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB.phased.vcf.gz"
output_vcf = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_R175K_filtered.vcf.gz"
snp_list_file = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_filtered_MAF_R175K_SNP_IDs.txt"
sample_list_file = "/home/jupyter/results/R11/Haplotype/RAB29_R175K_IDs_FID.IID.txt"

# Load list of SNP IDs to keep
with open(snp_list_file) as f:
    keep_ids = set(line.strip() for line in f if line.strip())

print(f"SNPs in ID list (MAF > 0.3): {len(keep_ids):,}")

# Load list of FID.IID sample names to keep
with open(sample_list_file) as f:
    lines = [line.strip() for line in f if line.strip()]
    # Try FID.IID format
    if len(lines[0].split()) == 1:
        keep_samples = lines
    else:
        keep_samples = [f"{line.split()[0]}.{line.split()[1]}" for line in lines]

print(f"Samples in carrier list: {len(keep_samples)}")

# Count total number of variants in original phased VCF
vcf_all = VCF(input_vcf)
n_total_variants = sum(1 for _ in vcf_all)
vcf_all.close()
print(f"Total variants in phased VCF: {n_total_variants:,}")

# Filter with sample and SNP list
vcf = VCF(input_vcf)
vcf.set_samples(keep_samples)

writer = Writer(output_vcf, vcf)
n_written = 0

for variant in vcf:
    if variant.ID in keep_ids:
        writer.write_record(variant)
        n_written += 1

writer.close()
vcf.close()

# Final summary
print(f"Variants written to filtered VCF: {n_written:,}")

In [ ]:
%%bash

cd /home/jupyter/results/R11/Haplotype/
tabix -p vcf RAB29_EUR_1MB_R175K_filtered.vcf.gz
gunzip -k RAB29_EUR_1MB_R175K_filtered.vcf.gz

In [ ]:
# Add columns to count number of different phased genotypes

# Input filtered VCF
vcf_path = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_R175K_filtered.vcf.gz"
output_tsv = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_R175K_filtered_table.tsv"

vcf = VCF(vcf_path)
samples = vcf.samples  # List of FID.IID sample names

rows = []

for variant in vcf:
    genotypes = variant.genotypes  # list of (allele1, allele2, phased) tuples

    # Convert to GT strings
    gt_strings = [
        f"{gt[0]}|{gt[1]}" if gt[2] else f"{gt[0]}/{gt[1]}"
        for gt in genotypes
    ]

    # Count phased genotype types
    count_00 = gt_strings.count("0|0")
    count_01 = gt_strings.count("0|1") + gt_strings.count("1|0")
    count_11 = gt_strings.count("1|1")

    # Build row: CHROM, POS, ID, REF, ALT, [genotypes...], counts
    row = [
        variant.CHROM,
        variant.POS,
        variant.ID,
        variant.REF,
        variant.ALT[0]  # assuming biallelic
    ] + gt_strings + [count_00, count_01, count_11]

    rows.append(row)

# Column names
columns = ["CHROM", "POS", "ID", "REF", "ALT"] + samples + ["0|0_Genotype", "0|1_or_1|0_Genotype", "1|1_Genotype"]

# Save as TSV
df = pd.DataFrame(rows, columns=columns)
df.to_csv(output_tsv, sep="\t", index=False)

print(f"TSV table written to: {output_tsv}")


In [ ]:
# Haplotype analysis
# Zoom in on the genomic region around R175K and keep only haplotype with R175K

# Set path to phased and filtered VCF and set zoom window
vcf_path = "/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_R175K_filtered.vcf.gz"
highlight_pos = 205770430
#zoom_window = 150000  # +/- window in bp around the highlight position
zoom_start = highlight_pos - 160000
zoom_end = highlight_pos + 70000

vcf = VCF(vcf_path)
samples = vcf.samples

hap1 = []
hap2 = []
positions = []
positions_numeric = []

for variant in vcf:
    pos = variant.POS
    if zoom_start <= pos <= zoom_end:
        positions.append(f"{variant.CHROM}:{pos}")
        positions_numeric.append(pos)
        genos = variant.genotypes
        hap1.append([g[0] for g in genos])
        hap2.append([g[1] for g in genos])

# Transpose and create DataFrames
hap1_df = pd.DataFrame(hap1).T
hap2_df = pd.DataFrame(hap2).T
hap1_df.columns = [f"SNP_{p}" for p in positions]
hap2_df.columns = [f"SNP_{p}" for p in positions]
hap1_df['Haplotype'] = 'allele1'
hap2_df['Haplotype'] = 'allele2'
hap_df = pd.concat([hap1_df, hap2_df])
hap_df['Sample'] = samples * 2
hap_df = hap_df.set_index(['Sample', 'Haplotype'])

# Find the SNP column corresponding to 205770430
target_col = None
for col in hap_df.columns:
    if col.startswith("SNP_") and f":{highlight_pos}" in col:
        target_col = col
        break

if target_col is None:
    raise ValueError(f"SNP at position {highlight_pos} not found in zoomed region.")

# Filter: Keep only haplotypes that carry the variant (allele == 1)
hap_df = hap_df[hap_df[target_col] == 1]

# Plotting
iids = [sample.split('.')[-1] for sample, hap in hap_df.index]
custom_yticks = [f"{iid}" for iid, (sample, hap) in zip(iids, hap_df.index)]

plt.figure(figsize=(12, 6))
#ax = sns.heatmap(hap_df.iloc[:, :-2], xticklabels=False, yticklabels=False,
                 #cmap=mcolors.ListedColormap(['#BCC9D9', '#003764']), cbar=False)
ax = sns.heatmap(hap_df.iloc[:, :-2], xticklabels=False, yticklabels=False,
                 cmap=mcolors.ListedColormap(['#D5DCE2', '#2485AC']), cbar=False)

# X-ticks
step = max(1, len(positions_numeric) // 20)
ax.set_xticks(range(0, len(positions_numeric), step))
ax.set_xticklabels([positions_numeric[i] for i in range(0, len(positions_numeric), step)],
                   rotation=90, fontsize=8)

# Y-ticks
ax.set_yticks(np.arange(len(custom_yticks)) + 0.5)
ax.set_yticklabels(custom_yticks, rotation=0, fontsize=8)

# Highlight variant
positions_array = np.array(positions_numeric)
closest_idx = (np.abs(positions_array - highlight_pos)).argmin()
ax.axvline(x=closest_idx, color='#A9001C', linestyle='--', linewidth=1)
ax.text(closest_idx, ax.get_ylim()[1] - 0.5, "p.R175K", ha='center', va='top',
        fontsize=9, color='#A9001C', fontweight='bold')

# Labels
plt.title("Disease haplotype (RAB29 p.R175K)", pad=20, fontweight='bold')
plt.xlabel("Genomic position (bp)", fontsize=12, fontweight='bold')
plt.ylabel("Sample Haplotype", fontsize=12, fontweight='bold')
plt.tight_layout()

# Save Plot
plt.savefig("/home/jupyter/results/R11/Haplotype/RAB29_EUR_1MB_R175K_filtered_haplotype_heatmap_filtered_zoom.png", dpi=300)

In [ ]:
# Determine overlap of haplotypes in zoomed in window
import numpy as np
from sklearn.metrics import pairwise_distances

# Uses haplotype with R175K variant
highlight_pos = 205770430

# Find SNP column
target_col = None
for col in hap_df.columns:
    if col.startswith("SNP_") and f":{highlight_pos}" in col:
        target_col = col
        break

if target_col is None:
    raise ValueError(f"SNP at position {highlight_pos} not found.")

# Filter haplotypes carrying the variant (allele == 1)
hap_df_variant = hap_df[hap_df[target_col] == 1]

# Convert hap_df to binary 0/1 matrix (assumes genotypes encoded as 0 or 1)
binary_data = (hap_df > 0).astype(int)

# Convert to numpy array for distance calculation
X = binary_data.values

# Compute pairwise Hamming distance matrix between haplotypes
dist_matrix = pairwise_distances(X, metric="hamming")

# Get list of sample-haplotype labels from index
hap_labels = binary_data.index.tolist()  # tuples like ('FID.IID', 'hap1')

# Print pairs of haplotypes that are identical or very similar (<20% differences)
threshold = 0.2

for i in range(len(hap_labels)):
    for j in range(i+1, len(hap_labels)):
        dist = dist_matrix[i, j]
        if dist == 0:
            print(f"{hap_labels[i]} and {hap_labels[j]} are identical haplotypes.")
        elif dist < threshold:
            print(f"{hap_labels[i]} and {hap_labels[j]} differ by {dist*100:.2f}% of SNPs.")

In [ ]:
%%bash
# Extract Variants in disease haplotype region

# Prepare list of variants in disease haplotype region
cd /home/jupyter/results/R11/Haplotype/
/home/jupyter/tools/plink/plink1.9 \
--bfile RAB29_EUR_1MB_filtered_MAF \
--chr 1 \
--from-bp 205610430 \
--to-bp 205840430 \
--make-bed \
--out RAB29_EUR_disease-haplotype-region

awk '{print $2}' RAB29_EUR_disease-haplotype-region.bim > RAB29_EUR_disease-haplotype-region_SNP_IDs.txt

## LD of *RAB29* p.R175K with RAB29 GWAS hit

In [ ]:
%%bash
# Check LD of RAB29 p.R175K and GWAS hit
cd /home/jupyter/results/R11/Haplotype/

/home/jupyter/tools/plink/plink2 \
--vcf RAB29_EUR_1MB.phased.vcf \
--make-pgen \
--out RAB29_EUR_1MB.phased

/home/jupyter/tools/plink/plink2 \
--pfile RAB29_EUR_1MB.phased \
--ld chr1:205770430:C:T chr1:205770138:C:T \
--out RAB29_R175K_LD_GWAS

In [ ]:
%%bash
# Check variant carriers with RAB29 p.R175K and GWAS hit
cd /home/jupyter/results/R11/Haplotype/

/home/jupyter/tools/plink/plink2 \
--vcf RAB29_EUR_1MB.phased.vcf \
--make-pgen \
--out RAB29_EUR_1MB.phased

/home/jupyter/tools/plink/plink2 \
--pfile RAB29_EUR_1MB.phased \
--snps chr1:205770430:C:T, chr1:205770138:C:T \
--export AD \
--out RAB29_R175K_GWAS

## Regression analysis of *RAB29* p.R175K with RAB29 GWAS hit

In [ ]:
%%bash

# Exclude individuals from NBA data that also have WGS data
cd /home/jupyter/results/
awk '{print $1, $2}' RAB29.fam > WGS_Samples.txt

/home/jupyter/tools/plink/plink1.9 \
  --bfile EUR.RAB29 \
  --remove WGS_Samples.txt \
  --make-bed \
  --out EUR.RAB29_noWGS

# Merge RAB28 WGS and NBA plink file sets
/home/jupyter/tools/plink/plink1.9 \
--bfile RAB29 \
--bmerge EUR.RAB29_noWGS \
--make-bed \
--out RAB29_WGS_NBA_EUR


In [ ]:
%%bash

# Extract SNPs of interest
cd /home/jupyter/results/
/home/jupyter/tools/plink/plink1.9 \
--bfile RAB29_WGS_NBA_EUR \
--snps chr1:205770430:C:T, chr1:205770138:C:T \
--keep-allele-order \
--recode A \
--out RAB29_WGS_NBA_EUR_R175K_GWAS


In [ ]:
%%bash

# Extract SNPs of interest
cd /home/jupyter/results/
/home/jupyter/tools/plink/plink1.9 \
--bfile RAB29_WGS_NBA_EUR \
--snps chr1:205770430:C:T, chr1:205770138:C:T \
--recode \
--make-bed \
--out RAB29_WGS_NBA_EUR_R175K_GWAS

In [ ]:
# Subsetting clinical data to keep only a few columns 

# Load clinical data
clinical_data_df_pheno = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)

# Define conditions
cond_PD_1 = (clinical_data_df_pheno['GP2_phenotype_for_qc'] == 'PD') & (clinical_data_df_pheno['GP2_phenotype'] == 'PD')
cond_PD_2 = ((clinical_data_df_pheno['GP2_phenotype_for_qc'] == 'Other') &
    ((clinical_data_df_pheno['GP2_PHENO'] == 'PD') | (clinical_data_df_pheno['GP2_PHENO'] == 'Affected_PD')) &
    (clinical_data_df_pheno['diagnosis'] != 'SWEDD') &
    (clinical_data_df_pheno['diagnosis'] != 'tremor')
)

cond_Con_1 = (clinical_data_df_pheno['GP2_phenotype_for_qc'] == 'Control')

# Assign group labels
clinical_data_df_pheno['group'] = 'Others'
clinical_data_df_pheno.loc[cond_PD_1 | cond_PD_2, 'group'] = 'PD'
clinical_data_df_pheno.loc[cond_Con_1, 'group'] = 'Control'

# Add "new_assigned_phenotype" column using to the clinical data overview
clinical_data_df_pheno['new_assigned_phenotype'] = 'Others'
clinical_data_df_pheno.loc[cond_PD_1 | cond_PD_2, 'new_assigned_phenotype'] = 'PD'
clinical_data_df_pheno.loc[cond_Con_1, 'new_assigned_phenotype'] = 'Control'

# Keep only required columns
clinical_data_df_Regression = clinical_data_df_pheno[['GP2ID', 'study', 'wgs_label', 'diagnosis', 'baseline_GP2_phenotype_for_qc', 'baseline_GP2_phenotype', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'age_at_diagnosis', 'age_at_death', 'age_at_last_follow_up', 'race_for_qc', 'family_history_for_qc', 'new_assigned_phenotype']].copy()
clinical_data_df_Regression = clinical_data_df_Regression[clinical_data_df_Regression["new_assigned_phenotype"].isin(["PD", "Control"])]

## Prepare overview file set
# Load the .raw file
RAB29_R175K_GWASSNP_df = pd.read_csv("/home/jupyter/results/R11/RAB29_WGS_NBA_EUR_R175K_GWAS.raw", sep=r"\s+")

# Ensure IDs are same type
RAB29_R175K_GWASSNP_df["IID"] = RAB29_R175K_GWASSNP_df["IID"].astype(str)
clinical_data_df_Regression["GP2ID"] = clinical_data_df_Regression["GP2ID"].astype(str)

# Select only the columns you need
RAB29_R175K_GWASSNP_subset = RAB29_R175K_GWASSNP_df[
    ["IID", "chr1:205770430:C:T_T", "chr1:205770138:C:T_C"]
]

# Merge
clinical_data_df_Regression = clinical_data_df_Regression.merge(RAB29_R175K_GWASSNP_subset, left_on="GP2ID", right_on="IID", how="left")

# Drop redundant IID column
clinical_data_df_Regression = clinical_data_df_Regression.drop(columns=["IID"])

# Add extra column with info if a sample carries both RAB29 variants
df = clinical_data_df_Regression
snp1 = df["chr1:205770430:C:T_T"]
snp2 = df["chr1:205770138:C:T_C"]

# convert dosage → presence/absence
snp1_present = (snp1 > 0).astype(int)
snp2_present = (snp2 > 0).astype(int)

clinical_data_df_Regression["RAB29_combined"] = np.where(snp1.isna() | snp2.isna(), np.nan, snp1_present + snp2_present)

# Add extra columns where both SNPs are only binary (no variant -> 0, heterozygous or homozygous variant -> 1)
df["chr1:205770430:C:T_T_binary"] = np.where(df["chr1:205770430:C:T_T"].isna(), np.nan, (df["chr1:205770430:C:T_T"] > 0).astype(int))
df["chr1:205770138:C:T_C_binary"] = np.where(df["chr1:205770138:C:T_C"].isna(), np.nan, (df["chr1:205770138:C:T_C"] > 0).astype(int))

# Export
clinical_data_df_Regression.to_csv(f'/home/jupyter/results/Clinical_WGS_NBA_RAB29.csv', index=False)


In [ ]:
%%writefile /home/jupyter/results/R11/RAB29_Regression.R

## Run regression analysis with RAB29 p.R175K and GWAS hit
# Load data
df <- read.csv("/home/jupyter/results/Clinical_WGS_NBA_RAB29.csv", check.names = FALSE)

# Clean phenotype:
# remove whitespace issues
df$new_assigned_phenotype <- trimws(df$new_assigned_phenotype)

# convert to binary outcome
df$new_assigned_phenotype <- ifelse(df$new_assigned_phenotype == "PD", 1, 0)
df$new_assigned_phenotype <- as.numeric(df$new_assigned_phenotype)

# Ensure correct types
df$biological_sex_for_qc <- as.factor(df$biological_sex_for_qc)
df$age_at_sample_collection <- as.numeric(df$age_at_sample_collection)
df$`chr1:205770138:C:T_T` <- 2 - df$`chr1:205770138:C:T_C` # So that the risk allele is better visualized (risk allele is major allele)
df$`chr1:205770430:C:T_T` <- as.numeric(df$`chr1:205770430:C:T_T`)
df$`chr1:205770138:C:T_T` <- as.numeric(df$`chr1:205770138:C:T_T`)
df$RAB29_combined <- as.numeric(df$RAB29_combined)

tapply(df$new_assigned_phenotype, df$`chr1:205770430:C:T_T`, mean)
tapply(df$new_assigned_phenotype, df$`chr1:205770138:C:T_T`, mean)

# Remove missing data
df_clean <- na.omit(df[, c("new_assigned_phenotype", "biological_sex_for_qc", "age_at_sample_collection", "chr1:205770430:C:T_T", "chr1:205770138:C:T_T", "RAB29_combined")])

# Check data before model
print(table(df$new_assigned_phenotype))
print(nrow(df))

print(table(df$`chr1:205770430:C:T_T`))
print(table(df$`chr1:205770138:C:T_T`))
print(table(df$RAB29_combined))

# Run glm models
model1 <- glm(new_assigned_phenotype ~ `chr1:205770430:C:T_T` + `chr1:205770138:C:T_T` +  biological_sex_for_qc + age_at_sample_collection, data = df, family = binomial())
model2 <- glm(new_assigned_phenotype ~ `chr1:205770430:C:T_T` * `chr1:205770138:C:T_T` +  biological_sex_for_qc + age_at_sample_collection, data = df, family = binomial())
model3 <- glm(new_assigned_phenotype ~ RAB29_combined + biological_sex_for_qc + age_at_sample_collection, data = df, family = binomial())

model4 <- glm(new_assigned_phenotype ~ `chr1:205770430:C:T_T` + `chr1:205770138:C:T_T`, data = df, family = binomial())
model5 <- glm(new_assigned_phenotype ~ `chr1:205770430:C:T_T` * `chr1:205770138:C:T_T`, data = df, family = binomial())
model6 <- glm(new_assigned_phenotype ~ RAB29_combined, data = df, family = binomial())

model7 <- glm(new_assigned_phenotype ~ `chr1:205770430:C:T_T`, data = df, family = binomial())
model8 <- glm(new_assigned_phenotype ~ `chr1:205770138:C:T_T`, data = df, family = binomial())

print(summary(model1))
print(summary(model2))
print(summary(model3))
print(summary(model4))
print(summary(model5))
print(summary(model6))
print(summary(model7))
print(summary(model8))

# Summary as output
write.csv(summary(model1)$coefficients, "/home/jupyter/results/Additive-model_covariates_results.csv")
write.csv(summary(model2)$coefficients, "/home/jupyter/results/Interaction-model_covariates_results.csv")
write.csv(summary(model3)$coefficients, "/home/jupyter/results/Combined-model_covariates_results.csv")
write.csv(summary(model4)$coefficients, "/home/jupyter/results/Additive-model_results.csv")
write.csv(summary(model5)$coefficients, "/home/jupyter/results/Interaction-model_results.csv")
write.csv(summary(model6)$coefficients, "/home/jupyter/results/Combined-model_results.csv")
write.csv(summary(model7)$coefficients, "/home/jupyter/results/R175K_results.csv")
write.csv(summary(model8)$coefficients, "/home/jupyter/results/GWAS-SNP_results.csv")

In [ ]:
# Run in Terminal: Rscript /home/jupyter/results/R11/RAB29_Regression.R 